# Grade 12 · Unit 3 — Evaluation metrics

> **SYNTHETIC** health table.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def data_file(dataset, filename):
    """Find datasets/<dataset>/clean/<filename> by walking up from this folder."""
    here = Path.cwd().resolve()
    for folder in [here, *here.parents]:
        p = folder / "datasets" / dataset / "clean" / filename
        if p.exists():
            return p
    raise FileNotFoundError(f"Could not find {filename}. Open this notebook from inside the data-science-nepal folder.")

h = pd.read_csv(data_file("health-synthetic", "health_synthetic.csv"))
FEATURES = ["antenatal_visits", "wealth_quintile", "mother_age"]
X, y = h[FEATURES], h["institutional_delivery"]
print(h.shape, "| share delivering in a facility:", round(float(y.mean()), 3))

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, accuracy_score
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=1, stratify=y)
m = LogisticRegression().fit(Xtr, ytr)
proba = m.predict_proba(Xte)[:, 1]
pred = (proba >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(yte, pred).ravel()
assert tn + fp + fn + tp == len(yte)
print(f"TN={tn} FP={fp} FN={fn} TP={tp}")

## Metrics by hand, then with the library

In [ ]:
precision = tp / (tp + fp); recall = tp / (tp + fn)
assert abs(precision - precision_score(yte, pred)) < 1e-12 and abs(recall - recall_score(yte, pred)) < 1e-12
print(f"accuracy {accuracy_score(yte, pred):.3f} | precision {precision:.3f} | recall {recall:.3f} | F1 {f1_score(yte, pred):.3f}")

## Moving the threshold trades one error for another
**Predict:** if we lower the threshold, will recall rise or fall?

In [ ]:
rows = []
for t in (0.2, 0.4, 0.5, 0.6, 0.8):
    p = (proba >= t).astype(int)
    rows.append((t, round(precision_score(yte, p, zero_division=0), 3), round(recall_score(yte, p), 3)))
pd.DataFrame(rows, columns=["threshold", "precision", "recall"])

**Make:** a clinic can only follow up 100 mothers a month. Which metric matters most and what threshold would you pick? Write the trade-off.